## Silver Layer — Garmin Activities
Cleans and standardises raw activities from the bronze layer.
Transforms units, extracts JSON fields, and formats timestamps.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.silver;

In [0]:
%sql
CREATE OR REPLACE TABLE garmin_lakehouse.silver.activities AS

SELECT
    -- IDs
    CAST(activityId AS STRING) as activity_id,
    CAST(ownerId AS STRING) as owner_id,
    CAST(deviceId AS STRING) as device_id,
    CAST(sportTypeId AS STRING) as sport_type_id,

    -- Activity Info
    activityName as activity_name,
    get_json_object(activityType, '$.typeKey') as activity_type_key,
    locationName as location_name,

    -- Timestamps
    TO_DATE(startTimeLocal) as start_date,
    DATE_FORMAT(startTimeLocal, 'HH:mm:ss') as start_time_local,
    CAST(DATE_FORMAT(startTimeLocal, 'H') AS INT) as start_hour,
    DATE_FORMAT(TIMESTAMPADD(SECOND, CAST(duration AS INT), TO_TIMESTAMP(startTimeLocal)), 'HH:mm:ss') as end_time_local,

    -- Distances & Durations
    ROUND(distance / 1000, 2) as distance_km,
    ROUND(duration / 60, 1) as duration_minutes,
    ROUND(elapsedDuration / 60, 1) as elapsed_duration_minutes,
    ROUND(movingDuration / 60, 1) as moving_duration_minutes,

    -- Elevation
    ROUND(elevationGain, 1) as elevation_gain_m,
    ROUND(elevationLoss, 1) as elevation_loss_m,
    ROUND(minElevation, 1) as min_elevation_m,
    ROUND(maxElevation, 1) as max_elevation_m,

    -- Speed
    ROUND(averageSpeed * 3.6, 2) as avg_speed_kmh,
    ROUND(maxSpeed * 3.6, 2) as max_speed_kmh,

    -- Average Pace (min/km)
    CASE
        WHEN averageSpeed > 0 THEN ROUND(1000.0 / (averageSpeed * 60), 2)
        ELSE NULL
    END as avg_pace_min_per_km,

    -- Average Pace formatted as MM:SS
    CASE
        WHEN averageSpeed > 0 THEN
            CONCAT(
                CAST(FLOOR(1000.0 / (averageSpeed * 60)) AS INT), ':',
                LPAD(CAST(FLOOR((1000.0 / (averageSpeed * 60) - FLOOR(1000.0 / (averageSpeed * 60))) * 60) AS INT), 2, '0')
            )
        ELSE NULL
    END as avg_pace_formatted,

    -- Max Pace (min/km)
    CASE
        WHEN maxSpeed > 0 THEN ROUND(1000.0 / (maxSpeed * 60), 2)
        ELSE NULL
    END as max_pace_min_per_km,

    -- Max Pace formatted as MM:SS
    CASE
        WHEN maxSpeed > 0 THEN
            CONCAT(
                CAST(FLOOR(1000.0 / (maxSpeed * 60)) AS INT), ':',
                LPAD(CAST(FLOOR((1000.0 / (maxSpeed * 60) - FLOOR(1000.0 / (maxSpeed * 60))) * 60) AS INT), 2, '0')
            )
        ELSE NULL
    END as max_pace_formatted,

    -- Heart Rate
    ROUND(averageHR, 0) as avg_heart_rate,
    ROUND(maxHR, 0) as max_heart_rate,

    -- Heart Rate Zones
    ROUND(hrTimeInZone_1, 0) as hr_zone_1_seconds,
    ROUND(hrTimeInZone_2, 0) as hr_zone_2_seconds,
    ROUND(hrTimeInZone_3, 0) as hr_zone_3_seconds,
    ROUND(hrTimeInZone_4, 0) as hr_zone_4_seconds,
    ROUND(hrTimeInZone_5, 0) as hr_zone_5_seconds,

    -- Running Metrics
    ROUND(averageRunningCadenceInStepsPerMinute, 0) as avg_cadence_spm,
    ROUND(maxRunningCadenceInStepsPerMinute, 0) as max_cadence_spm,
    ROUND(avgStrideLength, 2) as avg_stride_length_m,
    ROUND(steps, 0) as total_steps,

    -- Training Effect
    ROUND(aerobicTrainingEffect, 1) as aerobic_training_effect,
    ROUND(anaerobicTrainingEffect, 1) as anaerobic_training_effect,
    ROUND(vO2MaxValue, 1) as vo2_max,

    -- Calories
    ROUND(calories, 0) as total_calories,
    ROUND(bmrCalories, 0) as bmr_calories,

    -- Location
    ROUND(startLatitude, 6) as start_latitude,
    ROUND(startLongitude, 6) as start_longitude,

    -- Owner
    ownerFullName as owner_name,

    -- Metadata
    current_timestamp() as loaded_at

FROM garmin_lakehouse.raw.activities

In [0]:
%sql
SELECT 
    hr_zone_1_seconds,
    hr_zone_2_seconds,
    hr_zone_3_seconds,
    hr_zone_4_seconds,
    hr_zone_5_seconds
FROM garmin_lakehouse.silver.activities
LIMIT 5

In [0]:
%sql
SELECT *
FROM garmin_lakehouse.silver.activities
ORDER BY start_date DESC
LIMIT 10